# Phase 2A: SegFormer (Vision Transformer) Model Training & Fine-Tuning
### Urban Feature Extraction & Road Segmentation on Cartosat-1 (2.5m) Panchromatic Imagery

---

## 1. Architecture Overview: Why SegFormer for Urban Satellite Feature Extraction?
SegFormer is a state-of-the-art hierarchical Vision Transformer designed specifically for semantic segmentation:
1. **Hierarchical Transformer Encoder (MiT-B0)**: Generates multi-scale features at $1/4, 1/8, 1/16, 1/32$ of native resolution. This captures both fine road edges and broad urban land-use contexts without positional code interpolation issues.
2. **Lightweight All-MLP Decoder**: Efficiently aggregates multi-scale attention representations across all hierarchical levels.
3. **Pre-Trained Knowledge Transfer**: We initialize the encoder from **`nvidia/segformer-b0-finetuned-cityscapes-1024-1024`**, fine-tuning it to our 5 target classes.

### Target Classes:
* **Class 0**: Water (Blue)
* **Class 1**: Vegetation (Green)
* **Class 2**: Barren Land (Tan)
* **Class 3**: Built-up Urban Infrastructure (Red) - *Includes buildings, railway stations, rail tracks & yards*
* **Class 4**: Pure Roads (Yellow) - *Automotive & pedestrian street networks*


In [ ]:
# ==========================================
# 1. Imports and Environment Setup
# ==========================================
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add scripts directory to path
current_dir = Path.cwd()
scripts_dir = current_dir / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

from dataset import CartosatUrbanDataset
from model import build_model
from loss import CompoundRoadPrioritizedLoss
from evaluate import evaluate_dataset

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


---
## 2. Dataset Pipeline & Class Distribution
We load our **246 verified Cartosat-1 patches** ($512 \times 512$ px) from Phase 1.
Data augmentation (random horizontal/vertical flips, 90° rotations, brightness/contrast jitter) is applied to training patches.


In [ ]:
# ==========================================
# 2. Inspect Dataset Splits
# ==========================================
patches_dir = current_dir.parents[1] / "Phase_1_Data_Prep_and_Labeling" / "data" / "patches"

train_dataset = CartosatUrbanDataset(patches_dir, split="train", augment=True)
val_dataset = CartosatUrbanDataset(patches_dir, split="val", augment=False)
test_dataset = CartosatUrbanDataset(patches_dir, split="test", augment=False)

print(f"Training Patches:   {len(train_dataset)}")
print(f"Validation Patches: {len(val_dataset)}")
print(f"Test Patches:       {len(test_dataset)}")
print(f"Total Patches:      {len(train_dataset) + len(val_dataset) + len(test_dataset)}")


In [ ]:
# ==========================================
# 3. Sample Training Batch Preview
# ==========================================
from torch.utils.data import DataLoader
from evaluate import mask_to_rgb

sample_loader = DataLoader(train_dataset, batch_size=3, shuffle=True)
batch = next(iter(sample_loader))

pixel_vals = batch["pixel_values"].numpy()
labels = batch["labels"].numpy()

fig, axes = plt.subplots(len(pixel_vals), 3, figsize=(15, 5 * len(pixel_vals)), dpi=120)

mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

for i in range(len(pixel_vals)):
    raw = (pixel_vals[i] * std + mean).transpose(1, 2, 0)
    raw = np.clip(raw * 255.0, 0, 255).astype(np.uint8)
    mask_rgb = mask_to_rgb(labels[i])
    overlay = cv2.addWeighted(raw, 0.55, mask_rgb, 0.45, 0)
    
    axes[i, 0].imshow(raw[:, :, 0], cmap="gray")
    axes[i, 0].set_title(f"Augmented Satellite ({batch['patch_id'][i]})")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(mask_rgb)
    axes[i, 1].set_title("5-Class Ground Truth Mask")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Overlay (Clean Roads in Yellow)")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


---
## 3. Pretrained SegFormer Architecture & Adaptation
We initialize SegFormer with Cityscapes urban weights and project the MLP classifier to 5 classes:


In [ ]:
# ==========================================
# 4. Instantiate Adapted SegFormer Model
# ==========================================
model = build_model(num_classes=5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"SegFormer (MiT-B0) Initialized on: {device}")
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")


---
## 4. Model Fine-Tuning with Mixed Precision FP16
We train using **Focal Loss + Dice Loss** with road weight $w_{\text{road}} = 3.0$ and AdamW optimizer.
Checkpoints and validation metrics are logged per epoch.


In [ ]:
# ==========================================
# 5. Execute Training Engine
# ==========================================
from train import train_segformer

output_dir = current_dir

results = train_segformer(
    patches_dir=patches_dir,
    output_dir=output_dir,
    epochs=10,
    batch_size=4,
    learning_rate=1e-4,
)


---
## 5. Quantitative Evaluation & Test Set Performance
We evaluate the best checkpoint on the unseen Test Split.


In [ ]:
# ==========================================
# 6. Test Set Metrics & Curves
# ==========================================
test_m = results["test_metrics"]
print("=" * 45)
print("  SEGFORMER FINAL EVALUATION ON UNSEEN TEST SET")
print("=" * 45)
print(f"  Overall Accuracy (OA):   {test_m['overall_accuracy']*100:.2f}%")
print(f"  Mean IoU (mIoU):          {test_m['mIoU']*100:.2f}%")
print(f"  Mean F1-Score:           {test_m['mF1']*100:.2f}%")
print("-" * 45)
for k, v in test_m.items():
    if k.startswith("iou_"):
        cname = k.replace("iou_", "").replace("_", " ").title()
        print(f"  {cname:<16} IoU: {v*100:.2f}%")
print("=" * 45)

# Display training curves
curve_path = output_dir / "training_curves.png"
if curve_path.exists():
    img_curve = plt.imread(str(curve_path))
    plt.figure(figsize=(14, 5), dpi=140)
    plt.imshow(img_curve)
    plt.axis("off")
    plt.title("SegFormer Loss and Validation IoU Trajectory", fontsize=13, fontweight="bold")
    plt.show()


---
## 6. Qualitative Visual Results on Unseen Test Patches
We inspect SegFormer predictions side-by-side against Ground Truth labels:


In [ ]:
# ==========================================
# 7. Visual Predictions Display
# ==========================================
test_visuals = sorted((output_dir / "visual_results" / "test_visuals").glob("*.png"))

for p in test_visuals[:4]:
    im = plt.imread(str(p))
    plt.figure(figsize=(20, 5), dpi=140)
    plt.imshow(im)
    plt.axis("off")
    plt.show()
